# 4. SLURM

With `cluster_type="slurm"`, each call of a decorated function becomes a batch
job: clustrix connects to the login node over SSH, writes the job script,
runs `sbatch`, waits, and brings back the result. Everything from tutorial 3
about the connection, host keys, the remote environment and exceptions
applies unchanged. This tutorial covers what is specific to a scheduler:

- partition, cores, memory and time;
- a loop divided into one job per chunk, and watching the jobs;
- how time limits, refused submissions and killed jobs are reported;
- cancelling.

**About the outputs.** They were produced on a single-node SLURM test cluster
on the same computer as the notebook (login node `127.0.0.1:2222`, user
`clx`, one partition called `debug`, 4 CPUs). Queue waits and timings on your
cluster will differ.

In [1]:
import logging
import os
import subprocess
import threading
import time

import clustrix
from clustrix import cluster, configure, get_config

logging.basicConfig(level=logging.WARNING, format="%(name)s: %(message)s")

KEY = os.path.expanduser("~/.ssh/id_ed25519")


def timed(label, function, *args):
    started = time.perf_counter()
    result = function(*args)
    print(f"{label:<26} {time.perf_counter() - started:6.1f} s")
    return result


def on_login_node(command, keep=None):
    "Run a shell command on the login node: what you would type in a terminal there."
    known_hosts = os.path.expanduser("~/.ssh/known_hosts")
    ssh = ["ssh", "-p", "2222", "-i", KEY, "-o", f"UserKnownHostsFile={known_hosts}", "clx@127.0.0.1"]
    lines = subprocess.run(ssh + [command], capture_output=True, text=True).stdout.rstrip().splitlines()
    # keep: show the header and only the rows containing this text.
    print("\n".join(line for line in lines if keep is None or keep in line or "JOBID" in line.upper() or line.startswith("---")))

`on_login_node` is only for this tutorial, to show `squeue` and `sacct`
output next to the Python that caused it. The test cluster's account is shared
with other work, so some cells pass `keep=` to show only the rows that belong
to the example, recognised by their time limit.

## Configure

The host is the cluster's **login node**. The host key must already be in
`~/.ssh/known_hosts` (tutorial 3); here that was done beforehand.

In [2]:
# cluster-required: a SLURM login node
configure(
    cluster_type="slurm",
    cluster_host="127.0.0.1",
    cluster_port=2222,
    username="clx",
    key_file=KEY,
    remote_work_dir="~/clustrix-tutorial",
    default_partition="debug",
    replicate_local_environment=False,
)

on_login_node("sinfo")

PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST
debug*       up   infinite      1   idle vm


`remote_work_dir` must be on a filesystem that the compute nodes can see as
well as the login node, because the job reads its input from there and writes
its result there. A home or scratch directory normally is; a login node's
local `/tmp` is not.

## One job

The decorator's keywords are the resources of *one job*. A job can report
what the scheduler gave it through the environment variables SLURM sets.

In [3]:
@cluster(cores=2, memory="500MB", time="00:05:00")
def allocation():
    import os
    import socket

    return {
        "node": socket.gethostname(),
        "job id": os.environ.get("SLURM_JOB_ID"),
        "partition": os.environ.get("SLURM_JOB_PARTITION"),
        "cpus": os.environ.get("SLURM_CPUS_PER_TASK"),
        "directory": os.getcwd(),
    }


granted = timed("first job (builds env)", allocation)
granted

first job (builds env)       11.3 s


{'node': 'vm',
 'job id': '825',
 'partition': 'debug',
 'cpus': '2',
 'directory': '/home/clx/clustrix-tutorial/job_1795774740_b2a26d71'}

The scheduler's own record of that job:

In [4]:
fields = "JobID%7,JobName%21,Partition%9,ReqCPUS%7,ReqMem%7,Timelimit%10,State%10,Elapsed%9"
time.sleep(5)    # the accounting database is updated a moment after a job ends
on_login_node(f"sacct -X -j {granted['job id']} --format={fields}")

  JobID               JobName Partition ReqCPUS  ReqMem  Timelimit      State   Elapsed 
------- --------------------- --------- ------- ------- ---------- ---------- --------- 
    825            allocation     debug       2    500M   00:05:00  COMPLETED  00:00:01


| `@cluster(...)` | `sbatch` | default from |
|---|---|---|
| `cores=2` | `--cpus-per-task=2` | `default_cores` (4) |
| `memory="500MB"` | `--mem=500MB` | `default_memory` (`8GB`) |
| `time="00:05:00"` | `--time=00:05:00` | `default_time` (`01:00:00`) |
| `partition="debug"` | `--partition=debug` | `default_partition` |
| `gpus=1` | `--gres=gpu:1` | nothing: no GPU is requested |

The defaults ask for 4 CPUs, 8 GB and one hour. Ask for what the function
needs: on a busy cluster a smaller request starts sooner, and a request larger
than any node can satisfy never starts. No job in this tutorial asks for a
GPU; tutorial 2 describes `gpus` and says how far it has been tested.

`cores=-1` ("every CPU here") has no meaning for a batch job and is refused
before anything is submitted:

In [5]:
try:
    cluster(cores=-1)(allocation)()
except ValueError as error:
    print(error)

cores=-1 means every CPU available to this process, which a SLURM job cannot be asked for: give @cluster(cores=...) or default_cores the number of CPUs each job needs.


## A loop divided into jobs

A loop that can be divided (tutorial 2) becomes one job per chunk. Each job
gets the resources the decorator names, and one call starts at most
`max_parallel_jobs` jobs (default 100). With 8 iterations and
`max_parallel_jobs=4`, that is 4 jobs of 2 iterations.

The call below blocks until all jobs are done, so it is started in a thread
here, only to be able to run `squeue` while it is waiting.

In [6]:
@cluster(cores=1, memory="500MB", time="00:09:00")
def hash_chain(seeds, length):
    import os

    finals, jobs = {}, set()
    for seed in seeds:
        value = seed
        for step in range(length):
            value = (value * 6364136223846793005 + step) % 18446744073709551557
        finals[seed] = value
        jobs.add(os.environ.get("SLURM_JOB_ID"))
    return finals, sorted(jobs)


print(clustrix.explain(hash_chain))

hash_chain:
  line 6: `for seed in seeds` -- SPLIT
      collected in order: finals, jobs
      read-only (checked at run time): length, os
      private to each iteration: seed, step, value
  code before the loop runs in the caller
  code after the loop runs in the caller


In [7]:
configure(max_parallel_jobs=4)
logging.getLogger("clustrix.decorator").setLevel(logging.INFO)

outcome = {}


def run():
    started = time.perf_counter()
    outcome["result"] = hash_chain(list(range(8)), 20_000_000)
    outcome["seconds"] = time.perf_counter() - started


thread = threading.Thread(target=run)
thread.start()
time.sleep(14)
on_login_node("squeue -u clx --format='%.8i %.9P %.16j %.2t %.8M %.5C %.9l'", keep="9:00")
thread.join()

logging.getLogger("clustrix.decorator").setLevel(logging.WARNING)

clustrix.decorator: hash_chain: running its loop as 4 job(s) of up to 2 iteration(s).


   JOBID PARTITION             NAME ST     TIME  CPUS TIME_LIMI
     829     debug  hash_chain.body PD     0:00     1      9:00
     827     debug  hash_chain.body  R     0:01     1      9:00
     828     debug  hash_chain.body  R     0:01     1      9:00
     826     debug  hash_chain.body  R     0:03     1      9:00


`ST` is the job state: `R` running, `PD` pending (waiting for resources).
A job is named after the function it runs, and each chunk of a divided loop
is `<function>.body`. All four jobs here are `hash_chain.body`, so tell them
apart by their IDs.

In [8]:
finals, job_ids = outcome["result"]
print(f"finished in {outcome['seconds']:.1f} s; the iterations ran in jobs {job_ids}")
time.sleep(5)
on_login_node(f"sacct -X -j {','.join(job_ids)} --format={fields}")

finished in 26.1 s; the iterations ran in jobs ['826', '827', '828', '829']


  JobID               JobName Partition ReqCPUS  ReqMem  Timelimit      State   Elapsed 
------- --------------------- --------- ------- ------- ---------- ---------- --------- 
    826       hash_chain.body     debug       1    500M   00:09:00  COMPLETED  00:00:09 
    827       hash_chain.body     debug       1    500M   00:09:00  COMPLETED  00:00:09 
    828       hash_chain.body     debug       1    500M   00:09:00  COMPLETED  00:00:10 
    829       hash_chain.body     debug       1    500M   00:09:00  COMPLETED  00:00:09


For comparison, the same loop as a single job. `parallel=False` turns the
division off.

In [9]:
single = cluster(cores=1, memory="500MB", time="00:09:00", parallel=False)(hash_chain.__wrapped__)

one_job_finals, one_job_ids = timed("as one job", single, list(range(8)), 20_000_000)
print("jobs:", one_job_ids, "| same answer:", one_job_finals == finals)

as one job                   34.4 s
jobs: ['830'] | same answer: True


Notes on dividing into jobs:

- **Each chunk is a separate submission** (its own `sbatch`), not a job array. A loop of 10,000
  short iterations and `max_parallel_jobs=100` is 100 jobs of 100 iterations.
  Lower `max_parallel_jobs` if your cluster limits the number of queued jobs
  per user.
- **Queue wait counts.** If jobs wait ten minutes each to start, dividing a
  two-minute loop is a loss. Divide loops whose iterations take minutes or
  more.
- **Cores are per job.** `@cluster(cores=4)` with a divided loop asks for
  4 CPUs for every chunk. If the loop body is single-threaded, ask for 1.

### Code around the loop

When the code before the loop does real work, such as loading or generating
data, it runs as a job of its own and its results are passed to the chunk
jobs. `explain` says so in its last lines.

In [10]:
@cluster(cores=1, memory="1GB", time="00:07:00")
def bootstrap_means(replicates):
    import os

    import numpy as np

    population = np.random.default_rng(0).lognormal(size=2_000_000)   # before the loop
    means, jobs = [], set()
    for replicate in range(replicates):
        rng = np.random.default_rng([1, replicate])
        means.append(float(rng.choice(population, size=population.size).mean()))
        jobs.add(os.environ.get("SLURM_JOB_ID"))
    low, high = np.percentile(means, [2.5, 97.5])                     # after the loop
    return (float(low), float(high)), sorted(jobs)


print(clustrix.explain(bootstrap_means))

bootstrap_means:
  line 9: `for replicate in range(replicates)` -- SPLIT
      collected in order: jobs, means
      read-only (checked at run time): np, os, population
      private to each iteration: replicate, rng
  code before the loop runs as its own job
  code after the loop runs as its own job


In [11]:
began = time.strftime("%H:%M:%S")
interval, loop_jobs = timed("bootstrap, 6 jobs", bootstrap_means, 16)
print("95% interval for the mean:", tuple(round(x, 4) for x in interval))
print("loop ran in jobs:", loop_jobs)

bootstrap, 6 jobs            39.6 s
95% interval for the mean: (1.6488, 1.6528)
loop ran in jobs: ['832', '833', '834', '835']


In [12]:
time.sleep(5)
on_login_node(f"sacct -X -u clx -S {began} --format={fields}", keep="00:07:00")

  JobID               JobName Partition ReqCPUS  ReqMem  Timelimit      State   Elapsed 
------- --------------------- --------- ------- ------- ---------- ---------- --------- 
    831   bootstrap_means.pre     debug       1      1G   00:07:00  COMPLETED  00:00:01 
    832  bootstrap_means.body     debug       1      1G   00:07:00  COMPLETED  00:00:01 
    833  bootstrap_means.body     debug       1      1G   00:07:00  COMPLETED  00:00:00 
    834  bootstrap_means.body     debug       1      1G   00:07:00  COMPLETED  00:00:01 
    835  bootstrap_means.body     debug       1      1G   00:07:00  COMPLETED  00:00:01 
    836  bootstrap_means.post     debug       1      1G   00:07:00  COMPLETED  00:00:01


Six jobs, and their names say what each was: `bootstrap_means.pre` generated
`population`, the four `.body` jobs are the loop's (compare the IDs above),
and `bootstrap_means.post` computed the percentiles. Six jobs for a few seconds of arithmetic is
all overhead, as the elapsed times show; the arrangement pays when each chunk
runs for minutes. The data made by the first job is sent to each
chunk job with its arguments, so for very large intermediate data it is
better to write it to the cluster's filesystem in one decorated function and
read it in the loop.

## When a job does not succeed

### The time limit

A job that runs past `time` is killed by SLURM. The call then raises
`JobFailedError`, which is a `RuntimeError`, with the scheduler's state, the
job's directory and the end of its log. This job asks for one minute and
sleeps for ten.

In [13]:
@cluster(cores=1, memory="200MB", time="00:01:00")
def too_slow():
    import time

    time.sleep(600)


started = time.perf_counter()
try:
    too_slow()
except RuntimeError as error:
    print(f"after {time.perf_counter() - started:.0f} s, {type(error).__name__}:")
    print(error)

after 78 s, JobFailedError:
Job 837 failed (according to sacct: state=TIMEOUT exit_code=0:0 node=vm workdir=/home/clx/clustrix-tutorial/job_1795774867_fa03c292).
Its files and logs are in /home/clx/clustrix-tutorial/job_1795774867_fa03c292 on 127.0.0.1.
Its error log:
  slurmstepd-vm: error: *** JOB 837 ON vm CANCELLED AT 2026-11-27T10:22:10 DUE TO TIME LIMIT ***


SLURM enforces limits on its own schedule, so the job ended somewhat after
the minute was up. Raise `time=`; nothing else will help. A job that did not
succeed keeps its directory on the cluster, as the message says.

### A request the scheduler refuses

`sbatch` rejects some requests outright: an unknown partition, more memory
than any node has, a time limit above the partition's maximum. The call fails
immediately with `sbatch`'s own message.

In [14]:
try:
    cluster(cores=1, memory="200MB", time="00:05:00", partition="gpu-big")(allocation.__wrapped__)()
except RuntimeError as error:
    print(error)

sbatch did not accept the job. It said: 'sbatch: error: invalid partition specified: gpu-big\nsbatch: error: Batch job submission failed: Invalid partition name specified'. The job script is at /home/clx/clustrix-tutorial/job_1795774945_bfbce942/job.sh on the cluster.


### A job that dies

A Python exception comes back as itself with the remote traceback (tutorial
3). A process that is killed, which is what running out of memory looks like,
cannot raise anything, so this is a `JobFailedError` too. The message is a
few lines: how the job ended, where its files are, and the end of its error
log. The rest is on the exception: `job_id`, `outcome`, `remote_dir`, and the
whole log as `error_log`.

In [15]:
@cluster(cores=1, memory="200MB", time="00:05:00")
def killed():
    import os
    import signal

    os.kill(os.getpid(), signal.SIGKILL)


try:
    killed()
except RuntimeError as error:
    print(error)
    print()
    print("job_id:    ", error.job_id)
    print("outcome:   ", error.outcome)
    print("remote_dir:", error.remote_dir)
    print("error_log: ", len(error.error_log.splitlines()), "lines")

Job 838 failed (according to sacct: state=FAILED exit_code=9:0 node=vm workdir=/home/clx/clustrix-tutorial/job_1795774946_f13cd331).
Its files and logs are in /home/clx/clustrix-tutorial/job_1795774946_f13cd331 on 127.0.0.1.
The end of its error log (83 lines, all of them in this exception's .error_log):
  /var/spool/slurmd/job00838/slurm_script: line 171: 17909 Killed                  /home/clx/clustrix-tutorial/envs/py313_0ae49ae781c3/venv2_execution/bin/python -c "<the job program, 81 lines>"

job_id:     838
outcome:    failed
remote_dir: /home/clx/clustrix-tutorial/job_1795774946_f13cd331
error_log:  83 lines


On a real cluster, a job that exceeded its memory is shown by
`sacct -j <id>` with the state `OUT_OF_MEMORY`; raise `memory=`.

### Waiting too long

A call waits for its job for at most `job_wait_timeout` seconds (default
86400, one day), checking every few seconds at first and then every
`job_poll_interval` seconds (default 30). If your jobs can sit in the queue
for longer than a day, raise `job_wait_timeout`. If the notebook's kernel
stops while a job is queued, the job still runs on the cluster, but its
result is no longer collected.

## Cancelling

Interrupting the kernel (the stop button) during a divided loop cancels the
jobs of that loop whose results had not been collected yet. The single job of
an undivided call is not cancelled by the interrupt and keeps running on the
cluster. After interrupting, look at the queue and cancel what is left:

```bash
squeue -u $USER
scancel <jobid>            # one job
scancel -u $USER -n <name> # by job name, e.g. hash_chain.body
```

The lower-level `ClusterExecutor` submits without waiting and can cancel by
job ID. This is the same machinery `@cluster` uses.

In [16]:
# cluster-required: submits a real job
from clustrix import ClusterExecutor
from clustrix.utils import serialize_function


def sleeper():
    import time

    time.sleep(300)


with ClusterExecutor(get_config()) as executor:
    job_id = executor.submit_job(
        serialize_function(sleeper, (), {}),
        {"cores": 1, "memory": "200MB", "time": "00:10:00", "partition": "debug"},
    )
    time.sleep(5)
    print("submitted", job_id, "- status:", executor.get_job_status(job_id))
    on_login_node(f"squeue -j {job_id} --format='%.8i %.9P %.16j %.2t %.8M'")

    executor.cancel_job(job_id)
    time.sleep(5)
    print("\nafter cancel_job:")
    on_login_node(f"sacct -X -j {job_id} --format=JobID,State,Elapsed")

submitted 839 - status: running


   JOBID PARTITION             NAME ST     TIME
     839     debug          sleeper  R     0:03



after cancel_job:


JobID             State    Elapsed 
------------ ---------- ---------- 
839          CANCELLED+   00:00:03


## Using the cluster's software

Most clusters provide compilers, MPI and Python through environment modules,
and some provide a tuned conda environment. The settings are the ones from
tutorial 3:

```python
configure(
    module_loads=["python/3.13", "openblas"],          # names from `module avail`
    environment_variables={"OMP_NUM_THREADS": "4"},
    pre_execution_commands=["source /shared/project/setup.sh"],
)

@cluster(cores=4, environment="our-lab-env")           # an existing conda environment
def analysis(...):
    ...
```

(Not executed here: the test cluster has no modules or conda environments.)

Inside a job, `@cluster(cores=-1)` on a nested function, or any code that asks
"how many CPUs do I have", should follow the allocation.
`clustrix.loop_workers.available_cpus()` does: it honours
`SLURM_CPUS_PER_TASK`.

## If something looks wrong

| You see | It means |
|---|---|
| `sbatch did not accept ...` followed by sbatch's message | The request is invalid for this cluster: partition name, memory, time limit or account. `sinfo` lists partitions and their limits. |
| `JobFailedError` naming `TIMEOUT` | The job ran past `time=`. |
| `JobFailedError` naming `OUT_OF_MEMORY`, or a killed job | The job exceeded `memory=`. |
| `JobFailedError` with too little of the log to tell | The exception's `error_log` holds all of it, and `remote_dir` is where the job's files are. |
| The call seems to hang | The job is pending. `squeue -u $USER` shows it with state `PD` and the reason in the last column. |
| Jobs stay `PD` with reason `Resources` or `Priority` | Normal queueing. With reason `QOSMax...` or `AssocMax...` you have hit a per-user limit: lower `max_parallel_jobs`. |
| `cores=-1 means every CPU available to this process, which a SLURM job cannot be asked for` | Give a number. |
| The first job takes minutes, later ones seconds | The remote environment was built once (tutorial 3). |
| Result never arrives although `sacct` says `COMPLETED` | `remote_work_dir` is not on a filesystem shared between login and compute nodes. |

## Next

[Tutorial 5](05_data.ipynb): getting input files to the jobs.